In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import wavfile
import os
import tensorflow as tf

In [ ]:
# conjunto treino

# conjunto teste

### Separação da entrada
O conjunto de dados é dado no formato:

- final_0402_1:
	- dev_set:
	    - sount_event/train:
			- Classes:
				- Canais separados com diversos tipos de som para cada classe
		- test/oracle_target:
			- dev_test_*_Classe.wav
				- (esses arquivos contem os arquivos da pasta anterior mas agroupados em testes pelo numero e com tamanho de 10 segundos)

Para este trabalho vamos agrupar todas as classes, para que desta forma o modelo aprenda o comportamento do nosso subset escolhido mesmo quando apresentamos outros tipos de som parecidos. Também adicionaremos tipos diferentes de ruidos.

In [38]:
desired_classes = ['BicycleBell', 'Cough', 'MusicalKeyboard', 'Percussion', 'Doorbell']
base_path = './final_0402_1/DCASE2025Task4Dataset/dev_set/test/oracle_target'

all_groupped_by_test = {}
desired_tests = {}
test_audios = {}

for file_name in os.listdir(base_path):
	file_path = os.path.join(base_path, file_name)

	_, _, case, audioClass = file_name[:file_name.find(".")].split("_")
	if case not in all_groupped_by_test.keys():
		all_groupped_by_test[case] = [file_path]
	else:
		all_groupped_by_test[case] += [file_path]

for (case, paths) in all_groupped_by_test.items():
	classes = [w[w.rfind("_")+1:w.rfind(".")] for w in paths]
	if not any([cla in classes for cla in desired_classes]):
		continue
	desired_tests[case] = paths

for idx, (case, paths) in enumerate(desired_tests.items()):
	raw = [wavfile.read(path) for path in paths]
	sample_rates, audios = [r[0] for r in raw], [r[1] for r in raw]
	if any([sr != sample_rates[0] for sr in sample_rates]):
		print("Wrong sample rate on set")
		break
	combined_audio = audios[0]
	for i in range(1, len(audios)):
		combined_audio += audios[i]
	test_audios[case] = (sample_rates[0], combined_audio)
	target_file = os.path.join("tests", "case_%04d.wav" % (idx))
	if os.path.exists(target_file):
		os.remove(target_file)
	wavfile.write(target_file, sample_rates[0], combined_audio.astype(np.int16))
		

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

### Transfer Learning

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(include_top=False, weights='imagenet')

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Definir a entrada (128x128)
inputs = layers.Input(shape=(128, 128, 1))

# 2. Replicar o canal único 3 vezes para simular RGB, exigido pelo EfficientNetB0
x = layers.Concatenate()([inputs, inputs, inputs])

# 3. Aplicar o pré-processamento específico do EfficientNetB0
preprocess_input = tf.keras.applications.EfficientNetB0.preprocess_input
x = preprocess_input(x)

# 4. Base model (congelado)
base_model.trainable = False
x = base_model(x, training=False)

# 5. Cabeça de classificação (Top)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x) # Regularização para dataset pequeno
outputs = layers.Dense(10, activation='relu')(x)

model = models.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()